In [ ]:
# 
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [ ]:
passmark = 40

In [ ]:
%%time
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")
factor = 1000
df = pd.concat([df]*factor)
df.info()

In [ ]:
%%time
### cell 1 ###

df.isna().sum()

In [ ]:
%%time
### cell 2 ###

df.head()

In [ ]:
%%time
### cell 3 ###

df.describe()

In [ ]:
%%time
### cell 4 ###

df.isnull().sum()

In [ ]:
%%time
### cell 5 ###

# Convert once to a NumPy array and re‐use it for both assignments
arr = pd.to_numeric(df['math score'], errors='coerce').to_numpy()
df['math score'] = arr
df['Math_PassStatus'] = np.where(arr < passmark, 'F', 'P')
# Value counts unchanged:
df['Math_PassStatus'].value_counts()

In [ ]:
%%time
### cell 6 ###

df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')
df['Reading_PassStatus'] = np.where(df['reading score']<passmark, 'F', 'P')
df.Reading_PassStatus.value_counts()

In [ ]:
%%time
### cell 7 ###

df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')
df['Writing_PassStatus'] = np.where(df['writing score']<passmark, 'F', 'P')
df.Writing_PassStatus.value_counts()

In [ ]:
%%time
### cell 8 ###

# Vectorized evaluation of any failing status
mask = df[['Math_PassStatus', 'Reading_PassStatus', 'Writing_PassStatus']].eq('F').any(axis=1)
# Assign 'F' where any exam failed, otherwise 'P'
df['OverAll_PassStatus'] = mask.map({True: 'F', False: 'P'})
# Count the results
df['OverAll_PassStatus'].value_counts()

In [ ]:
%%time
### cell 9 ###

# Compute Total_Marks and Percentage in a single, vectorized assign call
df = df.assign(
    Total_Marks=lambda d: d[['math score', 'reading score', 'writing score']].sum(axis=1),
    Percentage=lambda d: d['Total_Marks'] / 3
)

In [ ]:
%%time
### cell 10 ###

# Vectorized grading without row-wise apply
# Define bins covering all possible Percentage values
bins = [-float('inf'), 40, 50, 60, 70, 80, float('inf')]
labels = ['F', 'E', 'D', 'C', 'B', 'A']

# Use pd.cut to assign grades based on Percentage, then force 'F' where OverAll_PassStatus=='F'
df['Grade'] = (
    pd.cut(df['Percentage'], bins=bins, labels=labels)
      .mask(df['OverAll_PassStatus'] == 'F', 'F')
      .astype(str)
)

# Get counts
df['Grade'].value_counts()